In [ ]:
%pip install matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create figures directory
os.makedirs("../reports/figures", exist_ok=True)

# Plot settings
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Load cleaned dataset
df = pd.read_csv("../data/processed/sales_cleaned.csv")

# to_datetime is use so that we can extract year , month , days
df["date"] = pd.to_datetime(df["date"])

print("Dataset loaded successfully")
print(df.shape)


In [ ]:
# Overall sales analysis
print("Total units sold:", df["sales"].sum())

print("Average daily product sales:", df["sales"].mean())

print("Median sales:", df["sales"].median())

print("Maximum daily product sales:", df["sales"].max())

print("Minimum daily product sales:", df["sales"].min())

In [ ]:
# Sales distribution
plt.figure(figsize=(10, 5))

sns.histplot(df["sales"], bins=30, kde=True)

plt.title("Distribution of Product Sales")
plt.xlabel("Units Sold")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# Daily aggregation
daily_sales = df.groupby("date")["sales"].sum().reset_index()

daily_sales.head()

In [ ]:
# Plot daily sales
plt.figure(figsize=(15, 5))

plt.plot(
    daily_sales["date"],
    daily_sales["sales"]
)

plt.title("Daily Total Sales Trend")
plt.xlabel("Date")
plt.ylabel("Total Units Sold")

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../reports/figures/01_daily_total_sales_trend.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Monthly sales analysis
monthly_sales = df.copy()

monthly_sales["year_month"] = (
    monthly_sales["date"].dt.to_period("M")
)

monthly_sales = (
    monthly_sales.groupby("year_month")["sales"]
    .sum()
    .reset_index()
)

monthly_sales["year_month"] = (
    monthly_sales["year_month"].astype(str)
)

monthly_sales.head()

In [ ]:
# Monthly sales visualization
plt.figure(figsize=(16, 5))

plt.plot(
    monthly_sales["year_month"],
    monthly_sales["sales"],
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Units Sold")

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("../reports/figures/02_monthly_sales_trend.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Store-wise sales analysis
store_sales = (
    df.groupby("store")["sales"]
    .sum()
    .sort_values(ascending=False)
)

print(store_sales)

In [ ]:
# Visualize store performance
plt.figure(figsize=(10, 5))

sns.barplot(
    x=store_sales.index,
    y=store_sales.values
)

plt.title("Total Sales by Store")
plt.xlabel("Store ID")
plt.ylabel("Total Units Sold")

plt.savefig("../reports/figures/03_store_sales_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Product-wise sales analysis
product_sales = (
    df.groupby("item")["sales"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 Products:")
print(product_sales.head(10))

print("\nBottom 10 Products:")
print(product_sales.tail(10))

In [ ]:
# Visualize top 10 products
top_products = product_sales.head(10)

plt.figure(figsize=(12, 5))

sns.barplot(
    x=top_products.index.astype(str),
    y=top_products.values
)

plt.title("Top 10 Best-Selling Products")
plt.xlabel("Product ID")
plt.ylabel("Total Units Sold")

plt.savefig("../reports/figures/04_top_10_products.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Visualize bottom 10 products
bottom_products = product_sales.tail(10)

plt.figure(figsize=(12, 5))

sns.barplot(
    x=bottom_products.index.astype(str),
    y=bottom_products.values
)

plt.title("Bottom 10 Lowest-Selling Products")
plt.xlabel("Product ID")
plt.ylabel("Total Units Sold")

plt.savefig("../reports/figures/05_bottom_10_products.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Weekly sales pattern
df["day_name"] = df["date"].dt.day_name()

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_sales = (
    df.groupby("day_name")["sales"]
    .mean()
    .reindex(weekday_order)
)

print(weekday_sales)

In [ ]:
# Visualize weekday sales
plt.figure(figsize=(10, 5))

sns.barplot(
    x=weekday_sales.index,
    y=weekday_sales.values
)

plt.title("Average Sales by Day of Week")
plt.xlabel("Day")
plt.ylabel("Average Units Sold")

plt.xticks(rotation=30)
plt.savefig("../reports/figures/06_average_sales_by_weekday.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Monthly seasonality analysis
df["month"] = df["date"].dt.month

month_sales = (
    df.groupby("month")["sales"]
    .mean()
)

print(month_sales)

In [ ]:
# Visualization
plt.figure(figsize=(10, 5))

sns.lineplot(
    x=month_sales.index,
    y=month_sales.values,
    marker="o"
)

plt.title("Average Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average Units Sold")

plt.xticks(range(1, 13))
plt.savefig("../reports/figures/07_average_sales_by_month.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Create a pivot table Store-product demand variation
store_item_sales = df.pivot_table(
    index="store",
    columns="item",
    values="sales",
    aggfunc="mean"
)

store_item_sales.head()


In [ ]:
# heatmap
plt.figure(figsize=(16, 7))

sns.heatmap(
    store_item_sales,
    cmap="YlGnBu"
)

plt.title("Average Product Sales Across Stores")
plt.xlabel("Product ID")
plt.ylabel("Store ID")

plt.savefig("../reports/figures/08_store_item_sales_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Save aggregated datasets for future use

daily_sales.to_csv(
    "../data/processed/daily_sales.csv",
    index=False
)

store_sales.to_csv(
    "../data/processed/store_sales.csv"
)

product_sales.to_csv(
    "../data/processed/product_sales.csv"
)

print("EDA aggregation files saved successfully")